# WTQ Data-Efficiency Experiment

This notebook asks whether normalized synthetic curriculum pretraining reduces the amount of labeled WikiTableQuestions data needed for fine-tuning. For each WTQ fraction, it compares **Base Qwen-LoRA** against **the normalized synthetic Level 3 checkpoint** using the exact same training subset, seed, optimizer settings, epochs, and official validation metric.

## 1. GPU and repository setup

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda} | available={torch.cuda.is_available()}")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

REPO_URL = "https://github.com/seungjun-green/cnn_qwen_table_mcr.git"
REPO_DIR = Path("/content/table-cnn-mrc")
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository")
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print(f"Git commit: {commit}")
%cd /content/table-cnn-mrc

## 2. Install dependencies and mount Drive

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)
from google.colab import drive, userdata
drive.mount("/content/drive")
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

## 3. Fair-comparison configuration

In [ ]:
WTQ_FRACTIONS = [0.10, 0.25]
INITIALIZATIONS = ["base", "curriculum"]
SUBSET_SEED = 2026
TRAINING_SEED = 42
EPOCHS = 6
LEARNING_RATE = 1e-4
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
EARLY_STOPPING_PATIENCE = 1
CHECKPOINT_EVERY_STEPS = 100
BASE_MODEL = "Qwen/Qwen3-1.7B"

DRIVE_ROOT = Path("/content/drive/MyDrive/cnn_qwen_table_mcr")
OUTPUT_ROOT = DRIVE_ROOT / "outputs/wtq_data_efficiency"
OFFICIAL_CACHE = DRIVE_ROOT / "outputs/diagnostics/wtq_official_1.0.2"
CURRICULUM_CHECKPOINT = (
    DRIVE_ROOT
    / "outputs/synthetic_curriculum_wtq_format/checkpoints/level_3/checkpoint.pt"
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if not CURRICULUM_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Missing curriculum checkpoint: {CURRICULUM_CHECKPOINT}")
print(f"WTQ fractions: {WTQ_FRACTIONS}")
print(f"Curriculum initialization: {CURRICULUM_CHECKPOINT}")
print(f"Direct Drive output: {OUTPUT_ROOT}")

## 4. Run the four paired conditions

The order is Base 10%, Curriculum 10%, Base 25%, Curriculum 25%. Each condition saves mid-epoch, best, and last checkpoints directly to Drive. Rerunning this cell resumes unfinished conditions and skips completed/early-stopped conditions.

In [ ]:
for fraction in WTQ_FRACTIONS:
    for initialization in INITIALIZATIONS:
        percentage = round(fraction * 100)
        run_name = f"wtq_{percentage:02d}pct_{initialization}"
        print("\n" + "#" * 88, flush=True)
        print(f"CONDITION: {run_name}", flush=True)
        print("#" * 88 + "\n", flush=True)
        command = [
            sys.executable, "-u", str(REPO_DIR / "scripts/run_data_efficiency.py"),
            "--output-root", str(OUTPUT_ROOT),
            "--official-cache-dir", str(OFFICIAL_CACHE),
            "--initialization", initialization,
            "--wtq-fraction", str(fraction),
            "--subset-seed", str(SUBSET_SEED),
            "--training-seed", str(TRAINING_SEED),
            "--epochs", str(EPOCHS),
            "--learning-rate", str(LEARNING_RATE),
            "--batch-size", str(BATCH_SIZE),
            "--gradient-accumulation-steps", str(GRAD_ACCUM_STEPS),
            "--checkpoint-every-steps", str(CHECKPOINT_EVERY_STEPS),
            "--early-stopping-patience", str(EARLY_STOPPING_PATIENCE),
            "--base-model", BASE_MODEL,
        ]
        if initialization == "curriculum":
            command.extend(["--curriculum-checkpoint", str(CURRICULUM_CHECKPOINT)])
        command_text = " ".join(shlex.quote(str(part)) for part in command)
        status_path = Path(f"/tmp/{run_name}_exit_code.txt")
        status_path.unlink(missing_ok=True)
        shell_command = (
            f"cd {shlex.quote(str(REPO_DIR))} && "
            f"PYTHONUNBUFFERED=1 TABLE_MRC_PLAIN_PROGRESS=1 "
            f"TABLE_MRC_PLAIN_LOG_EVERY=100 TQDM_DISABLE=1 "
            f"{command_text}; printf '%s' $? > {shlex.quote(str(status_path))}"
        )
        get_ipython().system(shell_command)
        if not status_path.is_file():
            raise RuntimeError(f"Exit status was not recorded for {run_name}")
        return_code = int(status_path.read_text(encoding="utf-8").strip())
        if return_code != 0:
            raise subprocess.CalledProcessError(return_code, command)

## 5. Verify paired subsets and compare results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

results_path = OUTPUT_ROOT / "results/data_efficiency_results.csv"
if not results_path.is_file():
    raise FileNotFoundError(results_path)
results = pd.read_csv(results_path).sort_values(
    ["wtq_percentage", "initialization"]
).reset_index(drop=True)
for percentage, pair in results.groupby("wtq_percentage"):
    if len(pair) == 2 and pair["subset_fingerprint"].nunique() != 1:
        raise RuntimeError(f"The {percentage}% paired runs used different subsets")
display(results[[
    "wtq_percentage", "initialization", "training_examples",
    "best_epoch", "best_validation_score",
    "curriculum_delta_vs_base", "status",
]])

pivot = results.pivot(
    index="wtq_percentage",
    columns="initialization",
    values="best_validation_score",
)
display(pivot)
ax = pivot.plot(kind="bar", figsize=(8, 4))
ax.set_xlabel("Percentage of WTQ training data")
ax.set_ylabel("Best WTQ validation denotation accuracy")
ax.set_title("Does synthetic curriculum improve WTQ data efficiency?")
ax.axhline(0.514659, color="gray", linestyle="--", label="WTQ 100% baseline: 0.5147")
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.xticks(rotation=0)
plt.show()

## Interpretation

The relevant number is `curriculum_delta_vs_base` at each WTQ percentage. A positive result means curriculum initialization helped while holding labeled WTQ data constant. Do not evaluate the WTQ test split during this comparison; validation is used to assess the data-efficiency hypothesis.